# **nn.Module로 XOR 리팩토링하기**

## **1. 데이터 준비**

**Q1. XOR을 학습시킬 데이터를 생성합니다.**

In [49]:
# code here
import torch 

X_tensor = torch.tensor([[0,0] ,[1,0], [0, 1], [1, 1]], dtype=torch.float32)
y_tensor = torch.tensor([0, 1, 1, 0], dtype=torch.float32).reshape(4,1)

In [50]:
X_tensor, y_tensor

(tensor([[0., 0.],
         [1., 0.],
         [0., 1.],
         [1., 1.]]),
 tensor([[0.],
         [1.],
         [1.],
         [0.]]))

## **2. 신경망 생성하기**

**Q1. MLP 클래스를 만든다.**

- __init__() : Linear Layer 2개와 sigmoid 레이어 1개를 생성한다.
  - fc1 : 2개의 input과 2개의 output을 가진 fully connected layer
  - fc2 : 2개의 input과 1개의 output을 가진 fully connected layer
  - sigmoid : sigmoid 활성화 함수 layer
- forward() :
  - fc1 layer -> sigmoid -> fc2 layer -> sigmoid

## **오버피팅 되지 않게 한번만 돌린다**

In [51]:
# code here
import torch.nn as nn

class MLP(nn.Module):


    def __init__(self): 
        super().__init__()    # requires_grad = True 같이 weight 직접 안만듦 -> 알아서 해줌
        
        # layer1 (입력 -> 은닉층)
        self.linear1 = nn.Linear(2, 2)
        #self.relu = nn.ReLU()
        self.sigmoid1 = nn.Sigmoid()

        # layer2 (은닉층 -> 출력층)
        self.linear2 = nn.Linear(2, 1)
        self.sigmoid2 = nn.Sigmoid()


    def forward(self, X):
        z1 = self.linear1(X)  # 일단 우리의 입력 데이터가 linear1 layer -> y=mx+b 가중합 구하기
        a1 = self.sigmoid1(z1) # 그리고 활성화
        z2 = self.linear2(a1) # 다음 층으로 값들을 전달
        a2 = self.sigmoid2(z2) # 그리고 활성화
        return a2             # 결과 (pred) return 

In [52]:
model = MLP()
y_pred = model(X_tensor)

In [53]:
print(y_pred)

tensor([[0.5545],
        [0.5521],
        [0.5693],
        [0.5666]], grad_fn=<SigmoidBackward0>)


## **3. Loss와 Gradient Descent**

**Q2. Loss 인스턴스를 만든다.**

criterion이라는 이름으로 nn.BCELoss의 인스턴스를 생성한다.

In [54]:
# code here
# 이진 분류
criterion = nn.BCELoss()
# 회귀 : MSELoss()
# 다중 분류: NLLLoss()

**Q3. Gradient Descent를 구현한다.**

- torch.no_grad() : 그래프에 기록되는것을 방지한다.
- model.parameters() : weight, bias등의 가중치를 반환한다.

In [ ]:
# code here

def gradient_descent(model, lr):
    with torch.no_grad():
        for p in model.parameters():
            # 이 p에다가 적용을 해준다.
            p -= p.grad * lr

## **4. 모델 학습**

**Q3. 모델을 학습시킨다.**
- learning rate : 1.0
- epoch : 1000

forward -> loss -> backward -> gradient descent 순으로 1개의 epoch를 완성한다.
- 한번의 epoch 이후 다음 epoch 시작 전, gradient 초기화를 해줘야 한다.

100번의 epoch마다 loss를 출력하게 한다.

model은 train모드로 사용해야 한다.

In [56]:
# code here
epochs = 1000
lr = 1.0


# model.train()
# 학습을 위해 epoch만큼 돌림
for epoch in range(epochs):
    model.train()  # 학습모드 -> model이 학습할 때 안에 파라미터가 바뀌지 않도록
    # 1. 순전파
    y_pred = model(X_tensor)   # 학습데이터
    # 2. loss구하기
    loss = criterion(y_pred, y_tensor)  # 머신러닝 때와 순서가 뒤바뀜(y_true, y_pred)
    # 3. loss를 기반으로 -> backpropagation
    loss.backward()
    # 4. gradietn descent
    gradient_descent(model, lr)  # 가중치 업데이트

    # gradient 초기화
    model.zero_grad()


    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch} : {loss}")
    

epoch 99 : 0.6925902962684631
epoch 199 : 0.6908305883407593
epoch 299 : 0.6768155694007874
epoch 399 : 0.5928080677986145
epoch 499 : 0.5027316212654114
epoch 599 : 0.19146263599395752
epoch 699 : 0.0736410915851593
epoch 799 : 0.043519049882888794
epoch 899 : 0.03053109161555767
epoch 999 : 0.023396223783493042


In [57]:
import numpy as np

np.where(y_pred > 0.5, 1, 0)

array([[0],
       [1],
       [1],
       [0]])

## **5. 결과**

**Q1. 모델을 eval 모드로, 예측 결과를 확인해보자.**

In [61]:
# code here
with torch.no_grad():
    y_pred = model(X_tensor)
    print(y_pred)
    print(np.where(y_pred >= 0.6, 1, 0))

tensor([[0.0271],
        [0.9777],
        [0.9805],
        [0.0234]])
[[0]
 [1]
 [1]
 [0]]
